In [2]:
from sqlalchemy import create_engine, Column, Integer, String, Float, DateTime
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker
from pathlib import Path
import pandas as pd

cwd = Path.cwd()
db_path = cwd / 'data/housing_market.db'
engine = create_engine(f'sqlite:///{db_path}?timeout=30', connect_args={'timeout': 30})
Session = sessionmaker(bind=engine)
Base = declarative_base()

class MedianPrice(Base):
    __tablename__ = 'median_prices'
    id = Column(Integer, primary_key=True)
    RegionID = Column(Integer)
    SizeRank = Column(Integer)
    RegionName = Column(String)
    RegionType = Column(String)
    StateName = Column(String)
    date = Column(DateTime)
    median_price = Column(Float)

class ZHVI(Base):
    __tablename__ = 'zhvi'
    id = Column(Integer, primary_key=True)
    RegionID = Column(Integer)
    SizeRank = Column(Integer)
    RegionName = Column(String)
    RegionType = Column(String)
    StateName = Column(String)
    date = Column(DateTime)
    zhvi = Column(Float)

class RedfinIndex(Base):
    __tablename__ = 'redfin_index'
    id = Column(Integer, primary_key=True)
    region_name = Column(String)
    date = Column(DateTime)
    redfin_hpi_yoy = Column(Float)
    redfin_hpi_mom = Column(Float)

class RedfinMarketTracker(Base):
    __tablename__ = 'redfin_market_tracker'
    id = Column(Integer, primary_key=True)
    period_begin = Column(DateTime)
    period_end = Column(DateTime)
    region = Column(String)
    median_sale_price = Column(Float)
    homes_sold = Column(Integer)
    new_listings = Column(Integer)
    inventory = Column(Integer)
    property_type = Column(String)
    last_updated = Column(DateTime)

# Create tables
Base.metadata.create_all(engine)
print("Database tables created successfully!")

Database tables created successfully!


/var/folders/m6/jw450gv97rx6cxclmnhdzppc0000gn/T/ipykernel_25097/1574124750.py:11: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [3]:
# Load CSV data into database
cwd = Path.cwd()
price_path = cwd / 'data' / 'Metro_median_sale_price_now_uc_sfrcondo_month.csv'
zhvi_path = cwd / 'data' / 'Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month (2).csv'
redfin_index_path = cwd / 'data' / 'Redfin_Home_Price_Index.csv'
redfin_tracker_path = cwd / 'data' / 'redfin_metro_market_tracker_small.tsv'

session = Session()

# Clear existing data
session.query(MedianPrice).delete()
session.query(ZHVI).delete()
session.query(RedfinIndex).delete()
session.query(RedfinMarketTracker).delete()
session.commit()

# Load Median Prices
price_df = pd.read_csv(price_path)
metadf_cols = ['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName']
price_long = price_df.melt(id_vars=metadf_cols, var_name='date', value_name='median_price')
price_long['date'] = pd.to_datetime(price_long['date'])

for _, row in price_long.iterrows():
    record = MedianPrice(
        RegionID=row['RegionID'],
        SizeRank=row['SizeRank'],
        RegionName=row['RegionName'],
        RegionType=row['RegionType'],
        StateName=row['StateName'],
        date=row['date'],
        median_price=row['median_price']
    )
    session.add(record)
session.commit()
print(f"Loaded {session.query(MedianPrice).count()} median price records")

# Load ZHVI
zhvi_df = pd.read_csv(zhvi_path)
zhvi_long = zhvi_df.melt(id_vars=metadf_cols, var_name='date', value_name='zhvi')
zhvi_long['date'] = pd.to_datetime(zhvi_long['date'])

for _, row in zhvi_long.iterrows():
    record = ZHVI(
        RegionID=row['RegionID'],
        SizeRank=row['SizeRank'],
        RegionName=row['RegionName'],
        RegionType=row['RegionType'],
        StateName=row['StateName'],
        date=row['date'],
        zhvi=row['zhvi']
    )
    session.add(record)
session.commit()
print(f"Loaded {session.query(ZHVI).count()} ZHVI records")

# Load Redfin Index
redfin_index = pd.read_csv(redfin_index_path, encoding='utf-16', sep='\t')
redfin_index['Date'] = pd.to_datetime(redfin_index['Month, Year of Date'], format='%B %Y')
redfin_index['redfin_hpi_yoy_numeric'] = redfin_index['Redfin HPI YoY'].str.rstrip('%').astype(float)
redfin_index['redfin_hpi_mom_numeric'] = redfin_index['Redfin HPI MoM'].str.rstrip('%').astype(float)

for _, row in redfin_index.iterrows():
    record = RedfinIndex(
        region_name=row['Region Name'],
        date=row['Date'],
        redfin_hpi_yoy=row['redfin_hpi_yoy_numeric'],
        redfin_hpi_mom=row['redfin_hpi_mom_numeric']
    )
    session.add(record)
session.commit()
print(f"Loaded {session.query(RedfinIndex).count()} Redfin Index records")

# load Redfin Market Tracker
tracker_df = pd.read_csv(redfin_tracker_path, sep='\t')
tracker_df["PERIOD_BEGIN"] = pd.to_datetime(tracker_df["PERIOD_BEGIN"])
tracker_df["PERIOD_END"] = pd.to_datetime(tracker_df["PERIOD_END"])
tracker_df["LAST_UPDATED"] = pd.to_datetime(tracker_df["LAST_UPDATED"], errors="coerce")
tracker_df["REGION"] = tracker_df["REGION"].str.replace(" metro area", "", regex=False).str.strip()

for _, row in tracker_df.iterrows():
    record = RedfinMarketTracker(
        period_begin=row["PERIOD_BEGIN"],
        period_end=row["PERIOD_END"],
        region=row["REGION"],
        median_sale_price=row.get("MEDIAN_SALE_PRICE"),
        homes_sold=row.get("HOMES_SOLD"),
        new_listings=row.get("NEW_LISTINGS"),
        inventory=row.get("INVENTORY"),
        property_type=row.get("PROPERTY_TYPE"),
        last_updated=row["LAST_UPDATED"]
    )
    session.add(record)
session.commit()
print(f"Loaded {session.query(RedfinMarketTracker).count()} Redfin Market Tracker records")

# Close session to prevent locking
session.close()
print("\nDatabase population complete! Session closed.")


Loaded 83709 median price records
Loaded 277450 ZHVI records
Loaded 8300 Redfin Index records
Loaded 500 Redfin Market Tracker records

Database population complete! Session closed.


In [4]:
# Create a fresh session for querying
session = Session()

# get median prices from database
price_records = session.query(MedianPrice).all()
price_df = pd.DataFrame([{
    'RegionID': r.RegionID,
    'SizeRank': r.SizeRank,
    'RegionName': r.RegionName,
    'RegionType': r.RegionType,
    'StateName': r.StateName,
    'date': r.date,
    'median_price': r.median_price
} for r in price_records])

# get ZHVI from database
zhvi_records = session.query(ZHVI).all()
zhvi_df = pd.DataFrame([{
    'RegionID': r.RegionID,
    'SizeRank': r.SizeRank,
    'RegionName': r.RegionName,
    'RegionType': r.RegionType,
    'StateName': r.StateName,
    'date': r.date,
    'zhvi': r.zhvi
} for r in zhvi_records])

In [5]:
# Avg median sale price over time
query1 = """
SELECT 
    date,
    AVG(median_price) AS avg_median_price
FROM median_prices
GROUP BY date
ORDER BY date;
"""

avg_price_over_time = pd.read_sql_query(query1, engine)
avg_price_over_time.head()

,date,avg_median_price
0,2008-02-29 00:00:00.000000,172918.492021
1,2008-03-31 00:00:00.000000,173497.167553
2,2008-04-30 00:00:00.000000,173159.050532
3,2008-05-31 00:00:00.000000,175155.462766
4,2008-06-30 00:00:00.000000,178150.840426


In [7]:
# Zillow median price + ZHVI
query2 = """
SELECT 
    mp.RegionName,
    mp.StateName,
    mp.date,
    mp.median_price,
    z.zhvi,
    (z.zhvi - mp.median_price) * 1.0 / mp.median_price AS rel_diff
FROM median_prices AS mp
JOIN zhvi AS z
    ON mp.RegionName = z.RegionName
   AND mp.date = z.date
WHERE mp.median_price IS NOT NULL AND z.zhvi IS NOT NULL;
"""

price_vs_zhvi = pd.read_sql_query(query2, engine)
price_vs_zhvi.head()


,RegionName,StateName,date,median_price,zhvi,rel_diff
0,United States,None,2008-02-29 00:00:00.000000,170500.0,201926.089818,0.184317
1,"New York, NY",NY,2008-02-29 00:00:00.000000,400000.0,458782.548499,0.146956
2,"Los Angeles, CA",CA,2008-02-29 00:00:00.000000,470000.0,530207.195443,0.128100
3,"Chicago, IL",IL,2008-02-29 00:00:00.000000,219000.0,247359.740119,0.129497
4,"Dallas, TX",TX,2008-02-29 00:00:00.000000,138000.0,156374.986393,0.133152


In [8]:
# ZHVI Bias by State
query3 = """
SELECT 
    mp.StateName,
    AVG( (z.zhvi - mp.median_price) * 1.0 / mp.median_price ) AS avg_bias,
    COUNT(*) AS samples
FROM median_prices AS mp
JOIN zhvi AS z
    ON mp.RegionName = z.RegionName
   AND mp.date = z.date
WHERE mp.median_price IS NOT NULL AND z.zhvi IS NOT NULL
GROUP BY mp.StateName
ORDER BY avg_bias DESC;
"""

state_bias = pd.read_sql_query(query3, engine)
state_bias.head()


,StateName,avg_bias,samples
0,PA,0.165554,3862
1,WI,0.162349,2672
2,NJ,0.150975,852
3,ND,0.139037,404
4,RI,0.134362,213


In [9]:
# Top 10 most expensive metros
query4 = """
SELECT 
    region,
    AVG(median_sale_price) AS avg_sale_price
FROM redfin_market_tracker
WHERE median_sale_price IS NOT NULL
GROUP BY region
ORDER BY avg_sale_price DESC
LIMIT 10;
"""

top10_expensive = pd.read_sql_query(query4, engine)
top10_expensive


,region,avg_sale_price
0,"San Francisco, CA",1410000.0
1,"Santa Cruz, CA",1010500.0
2,"Steamboat Springs, CO",993375.0
3,"Salinas, CA",902500.0
4,"New York, NY",850000.0
5,"Hood River, OR",790000.0
6,"Honolulu, HI",772500.0
7,"Santa Maria, CA",685000.0
8,"Los Angeles, CA",664850.0
9,"San Diego, CA",650000.0
